In [1]:
# SQLite = a tiny database that lives as a file on your computer
# sqlalchemy = Python library that connects pandas to databases
import pandas as pd
from sqlalchemy import create_engine, text

# Load your cleaned CSV
df = pd.read_csv('../data/cleaned/zomato_cleaned.csv')

# Create a SQLite database file called zomato.db
engine = create_engine('sqlite:///zomato.db')
# 'sqlite:///' means "create a local file database"

# Push the entire dataframe into a table called "restaurants"
df.to_sql('restaurants', engine,
          if_exists='replace',  # overwrite if table already exists
          index=False)

print("Database created!")
print(f"Table 'restaurants' has {len(df):,} rows")

Database created!
Table 'restaurants' has 51,717 rows


In [2]:
# Writing this once saves repetition for every query below
def run_query(sql):
    with engine.connect() as conn:
        result = pd.read_sql(text(sql), conn)
    return result

# Test it — list all column names in the table
test = run_query("SELECT * FROM restaurants LIMIT 2")
print(test.columns.tolist())

['url', 'address', 'name', 'online_order', 'book_table', 'rate', 'votes', 'phone', 'location', 'rest_type', 'dish_liked', 'cuisines', 'cost_for_two', 'reviews_list', 'menu_item', 'meal_type', 'city']


In [3]:
q1 = run_query("""
    SELECT location,
           COUNT(*) AS total_restaurants
    FROM restaurants
    GROUP BY location
    ORDER BY total_restaurants DESC
    LIMIT 10
""")
# COUNT(*) = count every row in that group
# GROUP BY = one result row per unique location
# ORDER BY DESC = highest count first
print(q1)

                location  total_restaurants
0                    BTM               5124
1                    HSR               2523
2  Koramangala 5th Block               2504
3               JP Nagar               2235
4             Whitefield               2144
5            Indiranagar               2083
6              Jayanagar               1926
7           Marathahalli               1846
8      Bannerghatta Road               1630
9              Bellandur               1286


In [4]:
q2 = run_query("""
    SELECT cuisines,
           ROUND(AVG(rate), 2) AS avg_rating,
           COUNT(*) AS total
    FROM restaurants
    WHERE cuisines IS NOT NULL
    GROUP BY cuisines
    HAVING total >= 100
    ORDER BY avg_rating DESC
    LIMIT 10
""")
# AVG() = average value of a column
# ROUND(x, 2) = round to 2 decimal places
# HAVING = like WHERE but runs AFTER grouping
# HAVING total >= 100 removes cuisines with very few restaurants
print(q2)

                         cuisines  avg_rating  total
0                  Cafe, Desserts        4.06    136
1             Desserts, Beverages        4.04    265
2               Cafe, Continental        3.89    173
3           North Indian, Mughlai        3.86    188
4             Desserts, Ice Cream        3.85    354
5                  Chinese, Momos        3.85    238
6             Ice Cream, Desserts        3.84    417
7                    Cafe, Bakery        3.83    110
8  North Indian, Chinese, Seafood        3.81    102
9       North Indian, Continental        3.79    136


In [5]:
q3 = run_query("""
    SELECT rest_type,
           ROUND(AVG(cost_for_two), 0) AS avg_cost,
           ROUND(AVG(rate), 2)         AS avg_rating,
           COUNT(*) AS total
    FROM restaurants
    WHERE rest_type IS NOT NULL
    GROUP BY rest_type
    HAVING total >= 200
    ORDER BY avg_cost DESC
    LIMIT 10
""")
print(q3)

             rest_type  avg_cost  avg_rating  total
0          Fine Dining    2708.0        4.15    346
1               Lounge    1694.0        3.90    396
2   Bar, Casual Dining    1334.0        4.10    425
3   Pub, Casual Dining    1261.0        4.06    255
4                  Bar    1254.0        3.74    697
5                  Pub    1243.0        3.97    357
6   Casual Dining, Bar    1237.0        4.06   1154
7  Casual Dining, Cafe     908.0        4.19    319
8        Casual Dining     789.0        3.74  10330
9                 Cafe     617.0        3.83   3732


In [6]:
q4 = run_query("""
    SELECT online_order,
           ROUND(AVG(rate), 2)         AS avg_rating,
           ROUND(AVG(cost_for_two), 0) AS avg_cost,
           COUNT(*) AS total_restaurants
    FROM restaurants
    GROUP BY online_order
""")
print(q4)

  online_order  avg_rating  avg_cost  total_restaurants
0           No        3.67     596.0              21273
1          Yes        3.72     525.0              30444


In [7]:
q5 = run_query("""
    SELECT name, location, votes, rate
    FROM restaurants
    WHERE location IN ('Koramangala 5th Block',
                         'Indiranagar', 'BTM')
    ORDER BY votes DESC
    LIMIT 10
""")
# IN (...) = match any value in the list
# Like saying: WHERE location = 'X' OR location = 'Y' OR location = 'Z'
print(q5)

       name               location  votes  rate
0      Toit            Indiranagar  14956   4.7
1      Toit            Indiranagar  14956   4.7
2  Truffles  Koramangala 5th Block  14726   4.7
3  Truffles  Koramangala 5th Block  14723   4.7
4  Truffles  Koramangala 5th Block  14723   4.7
5  Truffles  Koramangala 5th Block  14723   4.7
6  Truffles  Koramangala 5th Block  14717   4.7
7  Truffles  Koramangala 5th Block  14717   4.7
8  Truffles  Koramangala 5th Block  14710   4.7
9  Truffles  Koramangala 5th Block  14710   4.7


In [12]:
q6 = run_query("""
    SELECT name, location, votes, rate
    FROM restaurants
    WHERE location IN ('Koramangala 5th Block',
                         'Indiranagar', 'BTM')
    ORDER BY votes DESC
    LIMIT 10
""")
# IN (...) = match any value in the list
# Like saying: WHERE location = 'X' OR location = 'Y' OR location = 'Z'
print(q6)

       name               location  votes  rate
0      Toit            Indiranagar  14956   4.7
1      Toit            Indiranagar  14956   4.7
2  Truffles  Koramangala 5th Block  14726   4.7
3  Truffles  Koramangala 5th Block  14723   4.7
4  Truffles  Koramangala 5th Block  14723   4.7
5  Truffles  Koramangala 5th Block  14723   4.7
6  Truffles  Koramangala 5th Block  14717   4.7
7  Truffles  Koramangala 5th Block  14717   4.7
8  Truffles  Koramangala 5th Block  14710   4.7
9  Truffles  Koramangala 5th Block  14710   4.7


In [9]:
q7 = run_query("""
    SELECT name, location, cuisines,
           rate, cost_for_two, votes
    FROM restaurants
    WHERE rate >= 4.2
      AND cost_for_two <= 300
      AND votes >= 500
    ORDER BY rate DESC, votes DESC
    LIMIT 10
""")
# AND chains multiple conditions — all must be true
# ORDER BY two columns: first sort by rate, then break ties by votes
print(q7)

                   name      location              cuisines  rate  \
0                   CTR  Malleshwaram          South Indian   4.8   
1                   CTR  Malleshwaram          South Indian   4.8   
2  Brahmin's Coffee Bar  Basavanagudi          South Indian   4.8   
3    O.G. Variar & Sons   Rajajinagar      Bakery, Desserts   4.8   
4    O.G. Variar & Sons   Rajajinagar      Bakery, Desserts   4.8   
5                   CTR  Malleshwaram          South Indian   4.7   
6                   CTR  Malleshwaram          South Indian   4.7   
7          Taaza Thindi  Banashankari          South Indian   4.7   
8     Natural Ice Cream   Indiranagar  Ice Cream, Beverages   4.6   
9     Natural Ice Cream   Indiranagar   Ice Cream, Desserts   4.6   

   cost_for_two  votes  
0         150.0   4421  
1         150.0   4421  
2         100.0   2679  
3         200.0   1161  
4         200.0   1156  
5         150.0   4408  
6         150.0   4408  
7         100.0    651  
8         200.0

In [10]:
q8 = run_query("""
    SELECT book_table,
           ROUND(AVG(rate), 2)         AS avg_rating,
           ROUND(AVG(cost_for_two), 0) AS avg_cost,
           ROUND(AVG(votes), 0)         AS avg_votes,
           COUNT(*) AS total
    FROM restaurants
    GROUP BY book_table
""")
print(q8)

# Print a readable summary sentence
yes = q8[q8['book_table'] == 'Yes'].iloc[0]
no  = q8[q8['book_table'] == 'No'].iloc[0]
print(f"\nRestaurants WITH table booking:")
print(f"  Avg rating: {yes['avg_rating']} | Avg cost: ₹{yes['avg_cost']}")
print(f"\nRestaurants WITHOUT table booking:")
print(f"  Avg rating: {no['avg_rating']} | Avg cost: ₹{no['avg_cost']}")

  book_table  avg_rating  avg_cost  avg_votes  total
0         No        3.64     452.0      161.0  45268
1        Yes        4.13    1271.0     1147.0   6449

Restaurants WITH table booking:
  Avg rating: 4.13 | Avg cost: ₹1271.0

Restaurants WITHOUT table booking:
  Avg rating: 3.64 | Avg cost: ₹452.0


In [13]:
# Save each result so you can import them into Power BI later
q1.to_csv('../data/cleaned/sql_q1_locations.csv', index=False)
q2.to_csv('../data/cleaned/sql_q2_cuisines.csv', index=False)
q3.to_csv('../data/cleaned/sql_q3_rest_type.csv', index=False)
q4.to_csv('../data/cleaned/sql_q4_online_order.csv', index=False)
q5.to_csv('../data/cleaned/sql_q5_top_voted.csv', index=False)
q6.to_csv('../data/cleaned/sql_q6_location_rank.csv', index=False)
q7.to_csv('../data/cleaned/sql_q7_budget_gems.csv', index=False)
q8.to_csv('../data/cleaned/sql_q8_book_table.csv', index=False)

print("All 8 query results saved to data/cleaned/")

All 8 query results saved to data/cleaned/
